# Groupby and Arrest Data

In our merging exercises, we examined the relationship between county-level violent arrest totals and county-level drug arrest totals. In those exercises, you were given a dataset that provided you with county-level arrest totals. But that's not actually how the data is provided by the state of California. This week we will work with the *raw* California arrest data, which is not organized by county or even county-year. 



## Gradescope Autograding

Please follow [all standard guidance](https://practicaldatascience.org/ids720_specific/autograder_guidelines.html) for submitting this assignment to the Gradescope autograder, including storing your solutions in a dictionary called `results` and ensuring your notebook runs from the start to completion without any errors.

For this assignment, please name your file `exercise_groupby.ipynb` before uploading.

You can check that you have answers for all questions in your `results` dictionary with this code:

```python
assert set(results.keys()) == {
    "ex4_num_rows",
    "ex5_collapsed_vars",
    "ex7_alameda_1980_share_violent_arrestees_black",
    "ex11_white_drug_share",
    "ex11_black_drug_share",
    "ex12_proportionate",
}
```


### Submission Limits

Please remember that you are **only allowed three submissions to the autograder.** Your last submission (if you submit 3 or fewer times), or your third submission (if you submit more than 3 times) will determine your grade Submissions that error out will **not** count against this total.

### Exercise 1

Import the raw California arrest data from the State Attorney General's office. Please use [this link](https://github.com/nickeubank/MIDS_Data/blob/master/OnlineArrestData1980-2021.csv) (the original is here [here](https://openjustice.doj.ca.gov/data), but they keep updating it and I get tired of updating solutions, so... please use my copy!)

In [9]:
import pandas as pd
import numpy as np

results = {}

arrests = pd.read_csv("OnlineArrestData1980-2021.txt")

## Learning the Group Structure of Your Data

### Exercise 2

What is the unit of observation for this dataset? In other words, when row zero says that there were 505 arrests for `VIOLENT` crimes, what exactly is that telling you—505 arrests in 1980? 505 arrests in Alameda County?

(Please answer in Markdown)

In [25]:
arrests.head(30)

,YEAR,GENDER,RACE,AGE_GROUP,COUNTY,VIOLENT,PROPERTY,F_DRUGOFF,F_SEXOFF,F_ALLOTHER,F_TOTAL,M_TOTAL,S_TOTAL
0,1980,Male,Black,Under 18,Alameda County,505,1351,188,26,79,2149,2286,295
1,1980,Male,Black,18 to 19,Alameda County,205,465,183,8,48,909,1333,0
2,1980,Male,Black,20 to 29,Alameda County,949,1593,606,27,178,3353,7974,0
3,1980,Male,Black,30 to 39,Alameda County,450,755,241,18,110,1574,4876,0
4,1980,Male,Black,40 to 69,Alameda County,172,218,117,11,66,584,3836,0
5,1980,Male,Black,70 and over,Alameda County,4,2,2,0,0,8,79,0
6,1980,Male,Hispanic,Under 18,Alameda County,151,365,40,9,77,642,1303,187
7,1980,Male,Hispanic,18 to 19,Alameda County,83,89,48,0,58,278,963,0
8,1980,Male,Hispanic,20 to 29,Alameda County,226,243,158,13,86,726,3748,0
9,1980,Male,Hispanic,30 to 39,Alameda County,59,80,41,8,33,221,1669,0


> Each row is a year, gender, race, age_group, county summary for arrest statistics.
For example, the first row gives the arrest statistics for black, male, minors in Alameda County in the year 1980.

### Testing Your Assumptions

It's important to be able to test whether the data you are working with really is organized the way you think it is, especially when working with groupby. Let's discuss how to check your answer to Exercise 2 with the `.duplicated()` method. 

Consider the following toy data:

```python
import pandas as pd

df = pd.DataFrame(
    {
        "social_security_numbers": [
            111111111,
            222222222,
            222222222,
            333333333,
            333333333,
        ],
        "second_column": ["a", "a", "a", "a", "b"],
    }
)
df
```

If we want to see if there are any duplicate rows in the dataset, we can use the `.duplicated()` method:

```python
df.duplicated()
```

As you can see, `.duplicated()` looks at each row and returns `True` if it has seen the row it is looking at before. 

Notice that `.duplicted()` doesn't return `True` for *all* rows that look the same as another row in the data—it treats the first instance of a row as unique, and only tags *subsequent* repetitions as "duplicates" (You can change this behavior with the `keep` keyword argument if you want).

`.duplicated()` can also be used to test for rows that are duplicates in terms of a sub-set of columns. For example, `df.duplicated(["social_security_numbers"])` identifies rows with duplicate values of the variable `social_security_numbers`:

```python
df.duplicated(["social_security_numbers"])
```

Since `duplicated` is now only looking at `social_security_numbers` (not all values in a row), the last row is now a duplicate (because `333333333` is duplicated), where when we considered all columns, it was not a duplicate. 

### Testing with .duplicated()

`.duplicated()` can be paired with the `.any()` method to write `assert` tests to detect the presence of duplicates in our dataset and to validate our assumptions about whether certain variables uniquely identify observations in our data.

When you run `.any()` on an array of Booleans, it returns a single value of `True` if *any* entries are `True`, and a single value of `False` if *no* entries are `True`. (You can also use `.all()` to test if all entries are `False`).

Thus the command `df.duplicated(['social_security_numbers'])` will return `False` if `social_security_numbers` uniquely identifies every row in our dataset (since there are no duplicates)! But if there *are* any duplicates, it will return `True`:

```python
df.duplicated(["social_security_numbers"]).any()
```

So if I wanted a test that would raise an `Error` if it turned out that `social_security_numbers` *didn't* uniquely identify observations, I would combine that code with `assert not`:

```python
assert not df.duplicated(["social_security_numbers"]).any()

---------------------------------------------------------------------------
AssertionError                            Traceback (most recent call last)
Cell In [20], line 1
----> 1 assert not df.duplicated(["social_security_numbers"]).any()

AssertionError: 
```

(which in this case raises an Error because `social_security_numbers` *isn't* unique!)

BTW, if this feels oddly familiar, that's probably because we talked about this same idea when discussing merging. A `1:1` merge is one in which the merging variables uniquely identify observations in both datasets.

### Exercise 3

Use `duplicated` to test if the variables *you* think uniquely identify rows in your data really do uniquely identify rows. If you were wrong, update your beliefs (and your code) until you have an `assert` statement that passes.

In [ ]:
# year, gender, race, age_group, county

assert not arrests.duplicated(["YEAR", "GENDER", "RACE", "AGE_GROUP", "COUNTY"]).any()

### Exercise 4

Once you have a handle on how the data looks now, please **collapse the data** to be one observation per county-year-racial group. Store the number of rows in the resulting dataframe in your `results` dictionary under the key `"ex4_num_rows"`.

**Hint:** Think carefully about the most appropriate *aggregation function* (e.g., mean, median, sum, count) given the data we're working with!

**Hint 2:** When using `groupby`, I am a HUGE fan of the `as_index=False` keyword argument (e.g., `df.groupby("col1", as_index=False).mean()`). By default when you use groupby, pandas likes to make a new index out of the grouping columns. As you know, I HATE indices, and this keyword argument prevents pandas from this behavior. To illustrate:

In [ ]:
# remove age_group, gender

reduced_arrests = arrests.groupby(by=["YEAR", "RACE", "COUNTY"], as_index=False).agg(
    {
        "VIOLENT": "sum",
        "PROPERTY": "sum",
        "F_DRUGOFF": "sum",
        "F_SEXOFF": "sum",
        "F_ALLOTHER": "sum",
        "F_TOTAL": "sum",
        "M_TOTAL": "sum",
        "S_TOTAL": "sum",
    }
)

reduced_arrests.head(10)
# results["ex4_num_rows"] =

,YEAR,RACE,COUNTY,VIOLENT,PROPERTY,F_DRUGOFF,F_SEXOFF,F_ALLOTHER,F_TOTAL,M_TOTAL,S_TOTAL
0,1980,Black,Alameda County,2594,5138,1489,94,541,9856,24081,555
1,1980,Black,Butte County,32,33,5,0,7,77,139,0
2,1980,Black,Calaveras County,0,0,0,0,0,0,7,0
3,1980,Black,Colusa County,0,5,0,0,0,5,13,0
4,1980,Black,Contra Costa County,720,1422,368,24,257,2791,5201,40
5,1980,Black,Del Norte County,2,2,3,0,0,7,8,0
6,1980,Black,El Dorado County,12,19,1,1,3,36,58,0
7,1980,Black,Fresno County,519,867,97,14,84,1581,3096,107
8,1980,Black,Glenn County,1,1,1,1,0,4,8,0
9,1980,Black,Humboldt County,11,7,6,1,4,29,68,1


### Exercise 5

You should notice that there are significantly fewer rows in the data now. Given your answer from 3, what groups were you *collapsing* in question 4 (in other words, which different groups were previously split out into separate rows but have now been aggregated into the same rows)?

Store the names of those variables **in a `list` in alphabetical order** in `results` as `"ex5_collapsed_vars"`.

## Racial Arrest Composition

The next question we want to ask is: does the racial composition of arrests in each county vary by arrest type? In other words, do Blacks, Hispanics, Whites, and Other make up substantially different portions of the people arrested for drug offenses and violent offenses?

To answer this question, we need to be able to say what proportion of all drug arrestees were Black/White/Hispanic/Other and what proportion of all violent arrestees were Black/White/Hispanic/Other *for each county and for each year*.

In trying to do this, we'll need to break the problem down into pieces, starting from our goal (these quantities) and working backwards.

### Exercise 6

To calculate the share of all drug and violent arrestees who come from each racial group, what quantities do we need? 

Expressed differently, what two variables (one for drug arrests, one for violent arrests) do we need to add to our data to it allow us to calculate the share of arrestees in each county-year that come from a given racial group?

Calculate these two variables (one variable for drug arrests, one for violent arrests). 

**Hint:** `transform` should probably make an appearance here.

**Hint:** This types of groupby manipulations are an easy place to do things wrong, so it's important to eye-ball your data to be sure you did things right! 

Take a look at the data for, say, Alameda County in 1980, and compare the values that you calculate above with what you get if you try to calculate those same numbers by hand using the raw data from Exercise 5?

### Exercise 7

Now calculate the share of violent and drug arrestees in each county and year from each racial group.

To ensure you're on the right track, store the share (between zero and one) of violent arrestees in Alameda County in 1980 who were Black as `ex7_alameda_1980_share_violent_arrestees_black`.

### Exercise 8

We're about to start studying this data by plotting the share of violent arrestees that are Black against the share of felony drug arrestees that are Black. But the moment where you finish your data manipulations and are about to start you data analysis is a *great* time to just make sure everything in your data looks good. Let's run a few checks:

- Are your values of the share of felony arrestees who were arrested for violent crimes sensible?
- You're about to analyze the data using only the rows for the Black racial group. How many unique counties are there with data for Black arrestees? How many for White arrestees? Do you remember how many counties there are in CA (google is your friend if not!)?

Include assert statements to check all these features of your data.

(Reminder: the autograder will only work if you are entire notebook runs without errors.)

If you encounter any problems, make sure you understand why they are happening, and fix them (if a fix is appropriate!).

First, check arrest shares make sense:

OK, so something up with alpine county! Let's look at one of these years:

Ah, ok -- it's because no drug arrests! I guess np.nan is fine for those -- after all, I'm trying to calculate "share of drug arrests that were Black arrestees" -- if there are no drug arrests, then that's just undefined (not zero). Make sure that's only problem:

Now we can check if we have same county counts:

### Exercise 9

Now check to see if you have the same number of counties *in each year for each racial group*.

Can you figure out why there aren't? (Hint: this isn't trivial to figure out! You'll probably have to do some detective work, and even then you'll have to make a bit of a "best guess" at what's going on!)

Uh oh! Let's poke around to see what's up:

K, so we can see problem in 1980. Let's dig in:

K, Alpine obviously missing one. Let's look:

Huh, missing Hispanic and Black. Wonder why...

OK, digging around on the internet (and looking at the number of arrests Alpine reported), looks like Alpine county is a small and very White county, so perhaps they didn't have any Hispanic arrests, so didn't report them.

Indeed, that's also true for the other counties with fewer rows than they should have in 1984 -- if we go to [wikipedia](https://en.wikipedia.org/wiki/List_of_counties_in_California) and sort by population, we find these are some of the smallest counties:

Now, this isn't a slam dunk case -- it does appear counties sometimes report rows that are all zeros, even some of our smaller counties:

But this is the kind of detective work sometimes required for data science, and I think it's reasonable to infer that these are cases where there just were no arrests for a given racial group, and so nothing was reported. This is real world, government-agency-reported data -- there are gonna be issues.

### Exercise 10

Once you think you have a theory of the case, fix the problem. Again, this will NOT be trivial.

So let's fill in those cells with zeros, since without them we'd be systematically dropping observations where 0 of the arrests in a given category were of a given racial group (e.g. Black), and systematically dropping situations where the pct of arrestees who were Black would obviously bias our conclusions. 

Fixing this can actually be done in several ways, but here's what feels "intuitive" (nothing with groupby is intuitive, but this works) to me:

1. Identify the county-years missing Black entries (we're only working with Black and White later, so I don't need to worry about filling in zeros for Other and Hispanic. If I were working with all groups, I would).
2. Add in rows will all zeros for these entries.



**Part 1: Identify county-years missing Black entries**

Here we're going to use a trick that's actually something I use a lot: to figure out which groups contain observations of a given type, make a new variable that identifies which *rows* are of a type with a 1 (0 otherwise), then take the max of that variable for a group. Groups that include a row of the type we're looking for will have a max value of 1, those that don't will have a max value of 0. 

This works for finding, say, surveyed groups with men, or countries that experience a disaster in one year of your data, or customers whose purchases include the purchase of a given good:

**PART 2: Add in Black rows for these county-years**

Now that we've identified the county-years with no Black rows, we want to add in rows with zeros in all the cells we care about. Here there are quite a few solutions, so I'll do two (and you may have others!)

First, there are no county-years that lack a white row:

### Exercise 11

Now that you have corrected your problem, averaging over the values for each county-year, calculate the average share of drug arrestees who are Black and the average share of drug arrestees who are White. Store the results as `"ex11_white_drug_share"` and `"ex11_black_drug_share"`.

**Please round the shares to two decimal places (these are shares, so if one group was responsible for 50.1% of drug arrests, your rounded answer would be 0.50).**

### Exercise 12

Plot the share of violent arrestees that are Black against the share of felony drug arrestees that are Black. Do they look proportionate? Answer in text here.

Do Black arrestees make up a greater share of violent arrestees, or a greater share of drug arrestees? Store you answer as `"greater violent"`, or `"greater drug"` in `"ex12_proportionate"`.

**Hint:** You can add a 45 degree reference line by making a dummy dataset with one point at 0,0 and one point at a location near the top of your data (e.g. 0.7, 0.7). Without a reference line it's VERY hard to be sure if how your points relate to proportionality.

Proportionality is hard to see, so I'm gonna add a 45 degree line:

Clearly this is below the line of proportionality, meaning that Black arrestees make up a larger share of violent arrestees than drug arrests.

(A quick note of warning on interpretation: these results can tell you whether Black Californians make up a larger proportion of *arrests* for certain types of crimes, not whether they make up a larger proportion of people who *commit* a give type of crime! Those *might* be the same, but they might not... this data just can't answer that question.)